## 파이썬 게임

유니티 구현에 앞서, 유니티 개발까지 가지 못 했을 경우를 대비한 프로그램
구현할 것은 크게 5가지

1. 이미지 인식 처리와의 연동
2. 점수 증가 및 승자 처리
3. 멀티플레이
4. 찾아야할 것에 대한 알림
5. 현재 게임 상황에 대한 알림


## 라이브러리 임포트

In [82]:
%pip install torch torchvision torchaudio

Note: you may need to restart the kernel to use updated packages.


In [83]:
import cv2
import mediapipe as mp
import time
import pygame
import numpy as np

import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

from collections import Counter
from pathlib import Path

## 함수 정의

In [84]:
#이미지 인식 처리와의 연동
#지금은 이미지 인식 파트가 없으므로 키 입력 1, 0으로 성공 및 실패 여부를 판별한다.



In [85]:
#그래픽 관련 함수


def display_text_centered(text, font, color, surface):
    #화면 중앙에 글자를 띄우는 함수
    text_surface = font.render(text, True, color)
    text_rect = text_surface.get_rect(center=(surface.get_width() // 2, surface.get_height() // 2))
    surface.blit(text_surface, text_rect)

def draw_screen_border(surface, color, thickness):
    pygame.draw.rect(
        surface,
        color,
        surface.get_rect(),
        thickness
    )

def display_image(surface, image, x, y, scale):
    width = int(image.get_width() * scale)
    height = int(image.get_height() * scale)

    resized_image = pygame.transform.scale(
        image,
        (width, height)
    )

    image_rect = resized_image.get_rect(
        center=(x, y)
    )

    surface.blit(
        resized_image,
        image_rect
    )

def display_game_info(
    surface,
    font,
    color
):

    x = 20
    y = 20

    line_height = 35


    # 현재 라운드
    round_surface = font.render(
        f"Round : {current_round}",
        True,
        color
    )

    surface.blit(
        round_surface,
        (x, y)
    )

    y += line_height + 10


    # Player ID 순서대로 표시
    for pid in sorted(
        scores,
        key=lambda value: int(value)
    ):

        score = scores[pid]

        # JSON으로 받은 dict key는 문자열일 수 있음
        pid_int = int(pid)

        if pid_int == player_id:

            text = (
                f"Player{pid_int}(나) : {score}"
            )

        else:

            text = (
                f"Player{pid_int} : {score}"
            )


        text_surface = font.render(
            text,
            True,
            color
        )

        surface.blit(
            text_surface,
            (x, y)
        )

        y += line_height


# =========================
# 중앙 알림 표시
# =========================

def display_notice(surface):

    if time.time() >= notice_end_time:
        return

    if notice_title == "":
        return


    # 반투명 검은 배경
    overlay = pygame.Surface(
        (
            surface.get_width(),
            180
        ),
        pygame.SRCALPHA
    )

    overlay.fill(
        (0, 0, 0, 160)
    )


    overlay_y = (
        surface.get_height() // 2
        - 90
    )


    surface.blit(
        overlay,
        (
            0,
            overlay_y
        )
    )


    # 제목
    title_surface = default_Font.render(
        notice_title,
        True,
        (255, 255, 255)
    )

    title_rect = title_surface.get_rect(
        center=(
            surface.get_width() // 2,
            surface.get_height() // 2 - 30
        )
    )

    surface.blit(
        title_surface,
        title_rect
    )


    # 부제목
    if notice_subtitle:

        subtitle_surface = default_Font.render(
            notice_subtitle,
            True,
            (255, 255, 255)
        )

        subtitle_rect = subtitle_surface.get_rect(
            center=(
                surface.get_width() // 2,
                surface.get_height() // 2 + 30
            )
        )

        surface.blit(
            subtitle_surface,
            subtitle_rect
        )

In [86]:
#이미지 인식 및 점수 처리에 관한 함수

def update_score(is_success):
    #이미지 인식 성공 및 실패에 대한 점수 처리
    if(is_success):
        global score
        score += 1
        global gamePlayCount
        gamePlayCount += 1
        return True
    return False

In [87]:
#승자 처리에 관한 함수

#승자 처리
#현재 멀티가 없으므로 플레이어0만 존재

def update_winner(targetPlayerIndex):
    if(targetPlayerIndex == 0):
        winnerPlayerIndex = 0


In [88]:
#게임 로직에 관한 함수


from random import random


def getTargetImage():
    #5개의 이미지 중에서 랜덤으로 index를 반환한다.   
    return random.randint(0, 2)




## 변수 및 세팅

In [89]:
#여기서 필요한 변수을 선언

pygame.init()

running = False
default_Font = pygame.font.Font("../NotoSansKR-Regular.ttf",40)
default_Color = (255, 0, 0)

clock = pygame.time.Clock()

#게임 플레이
gamePlayCount = 0

#점수
score = 0
playrIndex = 0
winnerPlayerIndex = -1


successCapture = False

tryIndex = 0



In [90]:
# 카메라 세팅
cap = cv2.VideoCapture(
    0
)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)

MAX_CAMERA_RETRY = 5
CAMERA_RETRY_DELAY = 0.5

#앱 설정

WINDOW_NAME = "Python Game"

APP_WIDTH = 1280
APP_HEIGHT = 720

CAMERA_WIDTH = 1280
CAMERA_HEIGHT = 720

PANEL_X = 900
PANEL_WIDTH = 360



## 서버 관련

In [91]:
import socket
import threading
import json

player_id = -1
current_round = 0

round_winner = -1
game_winner = -1
answer_index = -1

already_sent = False



clients = {}
scores = {}


next_player_id = 1
current_round = 0
round_finished = False

SERVER_IP = "10.10.59.205"
PORT = 5000
WIN_SCORE = 5

game_lock = threading.Lock()


In [92]:
#서버관련 함수


# =========================
# 서버로 메시지 보내기
# =========================

def send_message(data):
    if not server_connected:
        return

    try:
        message = json.dumps(data) + "\n"

        client_socket.sendall(
            message.encode("utf-8")
        )

    except Exception as e:
        print("서버 전송 오류:", e)

# =========================
# 인식 성공 전송
# =========================

def send_detection():
    global already_sent

    if already_sent:
        return

    send_message({
        "type": "detected",
        "round": current_round
    })

    already_sent = True

    print(
        f"인식 성공 전송 / Round {current_round}"
    )
    
# =========================
# 서버 메시지 수신
# =========================

def receive_server():
    global player_id
    global current_round
    global scores

    global round_winner
    global game_winner

    global already_sent
    global server_connected
    global running

    global answer_index

    buffer = ""

    while server_connected:

        try:

            data = client_socket.recv(4096)

            if not data:
                print("서버 연결 종료")
                server_connected = False
                break
                
            buffer += data.decode("utf-8")

            while "\n" in buffer:

                line, buffer = buffer.split(
                    "\n",
                    1
                )

                if not line.strip():
                    continue

                message = json.loads(line)

                message_type = message.get(
                    "type"
                )


                # -------------------------
                # 서버 접속 완료
                # -------------------------

                if message_type == "connected":

                    player_id = message["player_id"]
                    current_round = message["round"]
                    scores = message["scores"]
                    answer_index = message["answer_index"]

                    print(
                        f"Player {player_id}로 접속"
                    )


                # -------------------------
                # 라운드 시작
                # -------------------------

                elif message_type == "round_start":

                    current_round = message["round"]

                    scores = message.get(
                        "scores",
                        scores
                    )
                    
                    answer_index = message["answer_index"]
                    round_winner = -1

                    # 다시 인식 가능
                    already_sent = False

                    print(
                        f"Round {current_round} 시작"
                    )

                    print(
                        f"정답 Index : {answer_index}"
                    )


                # -------------------------
                # 라운드 결과
                # -------------------------

                elif message_type == "round_result":

                    round_winner = message["winner"]
                    scores = message["scores"]

                    print(
                        f"Round {message['round']} "
                        f"승자 : Player {round_winner}"
                    )

                    print(
                        "현재 점수:",
                        scores
                    )


                # -------------------------
                # 게임 종료
                # -------------------------

                elif message_type == "game_over":

                    game_winner = message["winner"]
                    scores = message["scores"]

                    print(
                        f"게임 승자 : Player {game_winner}"
                    )

                    running = False


                # -------------------------
                # 플레이어 접속
                # -------------------------

                elif message_type == "player_joined":

                    print(
                        f"Player {message['player_id']} 접속"
                    )


                # -------------------------
                # 플레이어 연결 종료
                # -------------------------

                elif message_type == "player_left":

                    print(
                        f"Player {message['player_id']} 연결 종료"
                    )


        except Exception as e:

            if server_connected:
                print(
                    "서버 수신 오류:",
                    e
                )

            server_connected = False
            break

## AI 설정

In [93]:
#CNN 관련 세팅



# ==================================================
# 1. CNN 모델 정의
# ==================================================

class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ==================================================
# 2. 학습한 모델 불러오기
# ==================================================

checkpoint = torch.load(
    "../JUHA/models/object_cnn.pth",
    map_location="cpu"
)

classes = checkpoint["classes"]

model = CNN(len(classes))

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()


# ==================================================
# 3. 이미지 전처리
# ==================================================

transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.ToTensor()
])


# ==================================================
# 4. 게임 설정
# ==================================================

# 정답으로 인정할 최소 확률
confidence_threshold = 0.95

# 몇 번 연속 맞히면 정답인지
required_count = 20

# 현재 정답 연속 횟수
correct_count = 0

# 지금까지 맞힌 물건
completed_objects = []

for index, class_name in enumerate(classes):
    print(f"{index} : {class_name}")

0 : nipper
1 : pen
2 : wire stripper


## 메인 함수

In [94]:
print("게임 시작")

screen = pygame.display.set_mode((APP_WIDTH, APP_HEIGHT))
pygame.display.set_caption(WINDOW_NAME)

gamePlayCount = 0
running = True
answer_index = -1

notice_title = ""
notice_subtitle = ""

notice_end_time = 0

NOTICE_DURATION = 2.0


# =========================
# 카메라 세팅
# =========================

cap = cv2.VideoCapture(0)

cap.set(
    cv2.CAP_PROP_FRAME_WIDTH,
    1280
)

cap.set(
    cv2.CAP_PROP_FRAME_HEIGHT,
    720
)


# =========================
# 이미지 로드
# =========================

DIR = "../dataset"

images = [
    pygame.image.load(f"{DIR}/nipper/nipper_0000.jpg").convert_alpha(),
    pygame.image.load(f"{DIR}/pen/pen_0000.jpg").convert_alpha(),
    pygame.image.load(f"{DIR}/wire stripper/wire stripper_0000.jpg").convert_alpha(),
]

names = [
    "nipper",
    "pen",
    "wire stripper"
]

time.sleep(1.0)


# =========================
# 서버 연결
# =========================

client_socket = socket.socket(
    socket.AF_INET,
    socket.SOCK_STREAM
)

try:

    client_socket.connect(
        (
            SERVER_IP,
            PORT
        )
    )

    server_connected = True

    print("서버 연결 성공")

except Exception as e:

    server_connected = False

    print(
        "서버 연결 실패:",
        e
    )


# =========================
# 서버 수신 스레드 시작
# =========================

if server_connected:

    receive_thread = threading.Thread(
        target=receive_server,
        daemon=True
    )

    receive_thread.start()


# =========================
# 메인 루프
# =========================

while running:

    # -------------------------
    # 키 입력
    # -------------------------

    key_1_pressed = False

    for event in pygame.event.get():

        if event.type == pygame.QUIT:

            running = False


        if event.type == pygame.KEYDOWN:

            if event.key == pygame.K_e:

                running = False


            if event.key == pygame.K_1:

                key_1_pressed = True


    # -------------------------
    # 카메라
    # -------------------------

    successCapture, org_img = cap.read()


    if not successCapture or org_img is None:

        tryIndex += 1

        print(
            f"카메라 읽기 실패 "
            f"({tryIndex}/{MAX_CAMERA_RETRY})"
        )

        time.sleep(
            CAMERA_RETRY_DELAY
        )


        if tryIndex >= MAX_CAMERA_RETRY:

            print(
                "카메라 재연결 시도"
            )

            cap.release()

            time.sleep(0.5)

            cap = cv2.VideoCapture(0)

            cap.set(
                cv2.CAP_PROP_FRAME_WIDTH,
                1280
            )

            cap.set(
                cv2.CAP_PROP_FRAME_HEIGHT,
                720
            )

            tryIndex = 0

        continue


    tryIndex = 0


    # =========================
    # 카메라 그래픽 처리
    # =========================

    org_img = cv2.flip(
        org_img,
        1
    )

    org_img = cv2.cvtColor(
        org_img,
        cv2.COLOR_BGR2RGB
    )


    camera_surface = pygame.surfarray.make_surface(
        np.transpose(
            org_img,
            (1, 0, 2)
        )
    )


    camera_surface = pygame.transform.scale(
        camera_surface,
        (
            CAMERA_WIDTH,
            CAMERA_HEIGHT
        )
    )


    # =========================
    # 화면 초기화
    # =========================

    screen.fill(
        (30, 30, 30)
    )


    screen.blit(
        camera_surface,
        (0, 0)
    )


    # =========================
    # 문제 이미지
    # =========================

    if (
        answer_index >= 0
        and answer_index < len(images)
    ):

        display_image(
            screen,
            images[answer_index],
            APP_WIDTH - 300,
            APP_HEIGHT - 200,
            0.5
        )


    # =========================
    # 좌측 상단
    # 라운드 / 점수
    # =========================

    display_game_info(
        screen,
        default_Font,
        default_Color
    )


    # =========================
    # 화면 중앙 기본 문구
    # =========================

    display_text_centered(
        "물건을 찾아라",
        default_Font,
        default_Color,
        screen
    )


    # =========================
    # 라운드 시작 / 종료 알림
    # =========================

    display_notice(
        screen
    )


    # =========================
    # 화면 테두리
    # =========================

    draw_screen_border(
        screen,
        default_Color,
        10
    )


    # =========================
    # 테스트용 입력
    # =========================

    if (
        key_1_pressed
        and not already_sent
    ):

        send_detection()



    rgb = cv2.cvtColor(
        org_img,
        cv2.COLOR_BGR2RGB
    )

    # --------------------------------------------------
    # PIL 이미지
    # --------------------------------------------------

    image = Image.fromarray(rgb)


    # --------------------------------------------------
    # 이미지 크기 변경
    # --------------------------------------------------

    image = transform(image)


    # --------------------------------------------------
    # 배치 차원 추가
    # --------------------------------------------------

    image = image.unsqueeze(0)
    # ==================================================
    # AI 이미지 인식
    # ==================================================

    # org_img는 위에서 이미
    # cv2.COLOR_BGR2RGB로 변환된 상태임

    image = Image.fromarray(org_img)

    # 이미지 전처리
    image = transform(image)

    # 배치 차원 추가
    image = image.unsqueeze(0)


    # -------------------------
    # AI 예측
    # -------------------------

    with torch.no_grad():

        output = model(image)

        probability = torch.softmax(
            output,
            dim=1
        )

        confidence, predicted = torch.max(
            probability,
            dim=1
        )
        


    # -------------------------
    # 예측 결과
    # -------------------------

    label = classes[predicted.item()]

    confidence = confidence.item()


    # -------------------------
    # 서버에서 받은 정답 index
    # -------------------------

    if answer_index >= 0:

        target = classes[answer_index]

    else:

        target = ""


    # -------------------------
    # 정답 판정
    # -------------------------

    if (
        target != ""
        and label == target
        and confidence >= confidence_threshold
    ):

        correct_count += 1

    else:

        correct_count = 0


    # ==================================================
    # Pygame으로 AI 정보 표시
    # ==================================================

    ai_texts = [
        f"Target : {target}",
        f"AI : {label}",
        f"Confidence : {confidence * 100:.1f}%",
        f"Correct : {correct_count}/{required_count}"
    ]

    text_x = 20
    text_y = 180
    line_height = 35

    for text in ai_texts:

        text_surface = default_Font.render(
            text,
            True,
            default_Color
        )

        screen.blit(
            text_surface,
            (
                text_x,
                text_y
            )
        )

        text_y += line_height


    # ==================================================
    # 정답 성공
    # ==================================================

    print(f"Predicted {predicted.item() } Answer {answer_index} CorrectCount {correct_count} requiredCount {required_count} Already {already_sent}")
    if (correct_count >= required_count
        and not already_sent
    ):

        print("이미지 인식 성공")

        send_detection()

        correct_count = 0
        
    # =========================
    # 실제 이미지 인식 연결
    # =========================

    # detected = 이미지인식함수(org_img)
    #
    # if detected and not already_sent:
    #     send_detection()


    # =========================
    # 게임 종료
    # =========================

    if (
        game_winner != -1
        and time.time() >= notice_end_time
    ):

        running = False


    # =========================
    # 화면 갱신
    # =========================

    pygame.display.flip()


# =========================
# 종료
# =========================

cap.release()


try:

    client_socket.close()

except:

    pass


pygame.quit()

print("프로그램 종료")

게임 시작


[ WARN:0@1224.991] global cap_v4l.cpp:914 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[video4linux2,v4l2 @ 0x33b73700] ioctl(VIDIOC_G_INPUT): Inappropriate ioctl for device
[ERROR:0@1224.992] global obsensor_uvc_stream_channel.cpp:163 getStreamChannelGroup Camera index out of range


서버 연결 성공
Player 5로 접속
카메라 읽기 실패 (1/5)
Player 5 접속
Player 5 접속
카메라 읽기 실패 (2/5)
카메라 읽기 실패 (3/5)
카메라 읽기 실패 (4/5)
카메라 읽기 실패 (5/5)
카메라 재연결 시도
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Already False
Predicted 2 Answer 0 CorrectCount 0 requiredCount 20 Alread